# Add binding sites to specific membranes

In [ ]:
import gc
from pathlib import Path

from shapely.geometry import box, Point
from shapely.affinity import rotate

import cyanomembranes as cm
import matplotlib.pyplot as plt
import numpy as np

(OUT := Path("output")).mkdir(exist_ok=True)
import sys
from pathlib import Path

# Add the folder that CONTAINS membrane_analysis/ to the path
sys.path.insert(0, str(Path("..").resolve()))  # adjust if needed

## 0. Utilities

In [ ]:

def define_anchor_point(polygon, p1, p2):
    boundary = polygon.exterior

    d1 = boundary.project(Point(p1), normalized=True)
    d2 = boundary.project(Point(p2), normalized=True)

    anchor1 = np.array(boundary.interpolate(d1, normalized=True).coords[0])
    anchor2 = np.array(boundary.interpolate(d2, normalized=True).coords[0])

    arm1 = p1 - anchor1
    arm2 = p2 - anchor2

    centroid = np.array(polygon.centroid.coords[0])

    return {
        "d1": d1,
        "d2": d2,
        "arm1": arm1,
        "arm2": arm2,
        "anchor1_orig": anchor1,
        "anchor2_orig": anchor2,
        "centroid_orig": centroid
    }



def place_points(polygon, anchor_info, radius=5):

    d1, d2 = anchor_info["d1"], anchor_info["d2"]
    arm1, arm2 = anchor_info["arm1"], anchor_info["arm2"]

    anchor1_orig = anchor_info["anchor1_orig"]
    anchor2_orig = anchor_info["anchor2_orig"]

    centroid_orig = anchor_info["centroid_orig"]
    centroid_rot = np.array(polygon.centroid.coords[0])

    boundary_rot = polygon.exterior

    anchor1_rot = np.array(
        boundary_rot.interpolate(d1, normalized=True).coords[0]
    )

    anchor2_rot = np.array(
        boundary_rot.interpolate(d2, normalized=True).coords[0]
    )

    def rotate_arm(anchor_orig, anchor_rot, arm):

        v_orig = anchor_orig - centroid_orig
        v_rot  = anchor_rot  - centroid_rot

        angle = (
            np.arctan2(v_rot[1], v_rot[0])
            - np.arctan2(v_orig[1], v_orig[0])
        )

        R = np.array([
            [np.cos(angle), -np.sin(angle)],
            [np.sin(angle),  np.cos(angle)]
        ])

        return anchor_rot + R @ arm

    p1 = Point(*rotate_arm(anchor1_orig, anchor1_rot, arm1)).buffer(radius)
    p2 = Point(*rotate_arm(anchor2_orig, anchor2_rot, arm2)).buffer(radius)

    return p1, p2

def save_polygons_to_wkt(file_path, polygons) -> None:
    """Save polygons as WKT to a file"""
    with file_path.open("w") as f:
        for poly in polygons:
            f.write(poly.wkt + "\n")


## 1. Start with a specific complex

In [ ]:
PDB_FILES = [
    "1JB0-PSI-syn-cocc.pdb",
    "1NEK-SDH-Ecoli.pdb",
    "1OCO-cytoxidase-bov.trpdb",
    "1xl4-Kchannel-Pmagnetotacticum.trpdb",
    "3WU2-PSII-ThermosynVul.pdb",
    "4H13-cytb6f.trpdb",
]

DATA = Path("../data_cyano/")
(OUT := Path("../output")).mkdir(exist_ok=True)
(PIC_DIR :=  (OUT/"pictures_membranes")).mkdir(exist_ok=True)

proteins = cm.pdb_utils.process_proteins(DATA, OUT / "protein_shadows", PDB_FILES)

def get_complex_idx(p):
    psi_area = proteins["1JB0-PSI-syn-cocc"]["polygon"][0].area
    psii_area = proteins["3WU2-PSII-ThermosynVul"]["polygon"][0].area
    cytb6f_area = proteins["4H13-cytb6f"]["polygon"][0].area
    psii_idx = []
    psi_idx = []
    psimono_idx =  []
    cytb6f_idx = []
    for idx, i in enumerate(p):
        if np.isclose(i.area, psi_area):
            psi_idx.append(idx)
        if np.isclose(i.area, psii_area):
            psii_idx.append(idx)
        if np.isclose(i.area, cytb6f_area):
            cytb6f_idx.append(idx)
    return {"PSI": psi_idx, "PSII": psii_idx, "Cytb6f": cytb6f_idx}


## 2. Define binding sites

The binding sites must be derived by looking at structure file or must be guessed

In [ ]:
import matplotlib.pyplot as plt 

cytb6f = proteins['4H13-cytb6f']["polygon"][0]
p1 = Point(-40, 0).buffer(5) # gusessed binding site
p2 = Point(40, 0).buffer(5) # guessed binding site

fig,ax = plt.subplots()
ax.plot(*cytb6f.exterior.xy)
ax.plot(*p1.exterior.xy)
ax.plot(*p2.exterior.xy)
ax.set_aspect("equal")
plt.show()

## 3. Define anchor info

In [ ]:
anchor_info = define_anchor_point(cytb6f, np.array([-40, 0]), np.array([40, 0]))

## 4. Check if rotation works

In [ ]:
angles = [0, 30, 47, 90, 135]  # only used to CREATE test cases, not passed to place_points

fig, axes = plt.subplots(1, len(angles), figsize=(4 * len(angles), 4))

for ax, angle_deg in zip(axes, angles):
    cytb6f_rotated = rotate(cytb6f, angle_deg, origin='centroid')
    p1, p2 = place_points(cytb6f_rotated, anchor_info)   # no angle_deg !

    ax.plot(*cytb6f_rotated.exterior.xy, 'b-')
    ax.plot(*p1.exterior.xy, 'r-')
    ax.plot(*p2.exterior.xy, 'g-')
    ax.set_title(f'{angle_deg}°')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Test make membranes with binding sites

In [ ]:
p = cm.geo_utils.readwkt("../output/avg_membrane/polygons_avg_membrane_110-0.wkt")

idxes = get_complex_idx(p)
binding_sites = []

for idx in idxes["Cytb6f"][:10]:
    fig, ax = plt.subplots()
    cytb6f_rotated = p[idx]
    p1, p2 = place_points(cytb6f_rotated, anchor_info)
    binding_sites.extend([p1, p2])
    ax.plot(*cytb6f_rotated.exterior.xy, 'b-')
    ax.plot(*p1.exterior.xy, 'r-')
    ax.plot(*p2.exterior.xy, 'g-')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
p = cm.geo_utils.readwkt("../output/avg_membrane/polygons_avg_membrane_110-0.wkt")

idxes = get_complex_idx(p)
binding_sites = []

for idx in idxes["Cytb6f"]:
    cytb6f_rotated = p[idx]
    p1, p2 = place_points(cytb6f_rotated, anchor_info)
    binding_sites.extend([p1, p2])

p_new = p + binding_sites

fig, ax = plt.subplots()
for i in p_new:
    ax.plot(*i.exterior.xy, c="blue", lw=1)
ax.set_aspect("equal")
ax.set_xlim(300,450)
ax.set_ylim(200, 500)

# 6 Make membranes with binding sites

In [ ]:
OUT_BS = OUT/"avg_membrane_binding_sites"
OUT_BS.mkdir(exist_ok=True)

mem_file_lst = (OUT/"avg_membrane").glob("*110*")

for i in mem_file_lst:
    None
    p = cm.geo_utils.readwkt(i)

    idxes = get_complex_idx(p)
    binding_sites = []

    for idx in idxes["Cytb6f"]:
        cytb6f_rotated = p[idx]
        p1, p2 = place_points(cytb6f_rotated, anchor_info)
        binding_sites.extend([p1, p2])

    p_new = p + binding_sites
    save_polygons_to_wkt(OUT_BS/i.name, p_new)
    


In [ ]:
p = cm.geo_utils.readwkt("../output/avg_membrane_binding_sites/polygons_avg_membrane_110-0.wkt")


fig, ax = plt.subplots()
for i in p:
    ax.plot(*i.exterior.xy, c="blue", lw=1)
ax.set_aspect("equal")
ax.set_xlim(300,450)
ax.set_ylim(200, 500)

## 7 Make Simple analyses

In [ ]:
cytb6f_area = proteins["4H13-cytb6f"]["polygon"][0].area
psii_area   = proteins["3WU2-PSII-ThermosynVul"]["polygon"][0].area
bs_area = Point(1,1).buffer(5).area

cfg = cm.brownian_lattice.ExperimentLatticeConfig()

cfg.replicates = 3000
cfg.diff_coefficient = 3.5e9  # Å²/s == 3.5×10⁻⁷ cm²/s
cfg.particle_radius = 5
cfg.dimensions = (0, 5000)
cfg.random_start = True
cfg.has_ghost = True
cfg.store_history = False
cfg.workers = 10
cfg.nsteps = 100_000
cfg.save_every = 1000
cfg.chosen_obstacles_by_area = bs_area
cfg.steady_state_chosen_obstacle = True
cfg.start_area_by_size = (psii_area, 50, box(0, 0, 5000, 5000))

files = list((OUT_BS).glob("*60*"))
print(list(files))

ens_exp = cm.brownian_lattice.EnsembleExperimentLattice(files, cfg)
run = ens_exp.run()

In [ ]:
cfg2 = cm.brownian_lattice.ExperimentLatticeConfig()

cfg2.replicates = 3000
cfg2.diff_coefficient = 3.5e9  # Å²/s == 3.5×10⁻⁷ cm²/s
cfg2.particle_radius = 5
cfg2.dimensions = (0, 5000)
cfg2.random_start = True
cfg2.has_ghost = True
cfg2.store_history = False
cfg2.workers = 10
cfg2.nsteps = 100_000
cfg2.save_every = 1000
cfg2.chosen_obstacles_by_area = cytb6f_area
cfg2.steady_state_chosen_obstacle = True
cfg2.start_area_by_size = (psii_area, 50, box(0, 0, 5000, 5000))

files2 = list((OUT/"avg_membrane").glob("*60*"))
print(list(files2))

ens_exp2 = cm.brownian_lattice.EnsembleExperimentLattice(files2, cfg2)
run2 = ens_exp2.run()

In [ ]:
fig, ax = plt.subplots()
for i in [run, run2]:
    df_hits = i.get_mean_hits()
    hits_per_int = df_hits.drop("Time", axis=1)["Hits"]
    time = df_hits["Time"]
    dt = time.diff().iloc[1]
    rate_series = hits_per_int / (dt * cfg.replicates)
    k_mean = rate_series.iloc[-20:].mean()
    k_std = rate_series.iloc[-20:].std()
    
    ax.plot(time, rate_series)
ax.set_ylim(0, 60000)
ax.set_xlim(3e-6, )


## Resolution testing 

In [ ]:
cfg3 = cm.brownian_lattice.ExperimentLatticeConfig()

cfg3.lattice_resolution = 10
cfg3.replicates = 3000
cfg3.diff_coefficient = 3.5e9  # Å²/s == 3.5×10⁻⁷ cm²/s
cfg3.particle_radius = None #dont need here because one square is 10 A with is 1nm
cfg3.dimensions = (0, 5000)
cfg3.random_start = True
cfg3.has_ghost = True
cfg3.store_history = True
cfg3.workers = 10
cfg3.nsteps = 1000
cfg3.save_every = 10
cfg3.chosen_obstacles_by_area = cytb6f_area
cfg3.steady_state_chosen_obstacle = True
cfg3.start_area_by_size = (psii_area, 50, box(0, 0, 5000, 5000))

files3 = list((OUT/"avg_membrane").glob("*60*"))
print(list(files3))

ens_exp3 = cm.brownian_lattice.EnsembleExperimentLattice(files3, cfg3)
run3 = ens_exp3.run()

In [ ]:
fig, ax = plt.subplots()
for i in [run, run2, run3]:
    df_hits = i.get_mean_hits()
    hits_per_int = df_hits.drop("Time", axis=1)["Hits"]
    time = df_hits["Time"]
    dt = time.diff().iloc[1]
    rate_series = hits_per_int / (dt * cfg.replicates)
    k_mean = rate_series.iloc[-20:].mean()
    k_std = rate_series.iloc[-20:].std()
    
    ax.plot(time, rate_series)
ax.set_ylim(0, 60000)
ax.set_xlim(3e-6, )


In [ ]:
fig, ax = plt.subplots()
run3.runs[0].plot_run(ax=ax)
ax.set_xlim(0,100)
ax.set_ylim(0,100)

# Some testing

In [ ]:
TEST = OUT/"test"
TEST.mkdir(exist_ok="True")

from shapely.affinity import translate

p = cm.geo_utils.readwkt("../output/avg_membrane_binding_sites/polygons_avg_membrane_110-0.wkt")
idxes = get_complex_idx(p)

psii = translate(p[idxes["PSII"][1]], 500)
cytb6f_new = translate(p[idxes["Cytb6f"][1]], 0, 500)
circ1 = Point(750, 3200).buffer(700)
circ2 = Point(750, 3200).buffer(690)
ring = circ1.difference(circ2)

fig, ax = plt.subplots()
ax.plot(*psii.exterior.xy)
ax.plot(*cytb6f_new.exterior.xy)
ax.plot(*ring.exterior.xy)
ax.set_aspect("equal")

p_new = [psii, cytb6f_new, ring]

for i in range(10):
    save_polygons_to_wkt(TEST/f"mock_{i}.wkt", p_new)

p1, p2 = place_points(cytb6f_new, anchor_info)

fig, ax = plt.subplots()
ax.plot(*psii.exterior.xy)
ax.plot(*p1.exterior.xy)
ax.plot(*p2.exterior.xy)
ax.plot(*ring.exterior.xy)
ax.plot(*cytb6f_new.exterior.xy)
ax.set_aspect("equal")

p_new = [psii, cytb6f_new, ring, p1, p2]

for i in range(10):
    save_polygons_to_wkt(TEST/f"mock_{i}_bs.wkt", p_new)

In [ ]:
cytb6f_area = proteins["4H13-cytb6f"]["polygon"][0].area
psii_area   = proteins["3WU2-PSII-ThermosynVul"]["polygon"][0].area
bs_area = Point(1,1).buffer(5).area

cfg = cm.brownian_lattice.ExperimentLatticeConfig()

cfg.replicates = 1
cfg.diff_coefficient = 3.5e9  # Å²/s == 3.5×10⁻⁷ cm²/s
cfg.particle_radius = 5
cfg.dimensions = (0, 5000)
cfg.random_start = True
cfg.has_ghost = True
cfg.store_history = False
cfg.workers = 10
cfg.nsteps = 1000
cfg.save_every = 10
cfg.chosen_obstacles_by_area = bs_area
cfg.steady_state_chosen_obstacle = True
cfg.start_area_by_size = (psii_area, 100, box(0, 0, 5000, 5000))

files = list((TEST).glob("*bs*"))
print(list(files))

ens_exp = cm.brownian_lattice.EnsembleExperimentLattice(files, cfg)
run = ens_exp.run()

In [ ]:
cfg = cm.brownian_lattice.ExperimentLatticeConfig()

cfg.replicates = 300
cfg.diff_coefficient = 3.5e9  # Å²/s == 3.5×10⁻⁷ cm²/s
cfg.particle_radius = 5
cfg.dimensions = (0, 5000)
cfg.random_start = True
cfg.has_ghost = True
cfg.store_history = True
cfg.workers = 10
cfg.nsteps = 900_000
cfg.save_every = 10
cfg.chosen_obstacles_by_area = bs_area
cfg.steady_state_chosen_obstacle = True
cfg.start_area_by_size = (psii_area, 10, box(0, 0, 5000, 5000))

cfg.shapes = cm.geo_utils.readwkt('../output/test/mock_7_bs.wkt')
exp = cm.brownian_lattice.ExperimentLattice(cfg)
run = exp.run()


In [ ]:
fig, ax = plt.subplots()
run.plot_run(ax=ax)
ax.set_xlim(1000, 1500)
ax.set_ylim(2750, 3200)

# 